In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import openpyxl

import sys
import os
import runpy

import requests
from io import StringIO
import json

import json
import glob
from pathlib import Path

import pickle
import re
import glob

In [2]:
fpath = r"C:\Users\jarem\OneDrive - London School of Economics\YEAR 2\1. Policy paper\policy-paper-repo\data\clean\treatment\eu_flows\final"

In [3]:
from processing_functions import knit_directory_with_periods, get_period_from_filename

In [5]:
# # --- EXECUTE ---
# PATH_TO_OUTPUTS = r"C:\Users\jarem\OneDrive - London School of Economics\YEAR 2\1. Policy paper\policy-paper-repo\data\clean\treatment\eu_flows\final"  # <--- UPDATE THIS PATH

# df_powiat_all, df_gmina_all = knit_directory_with_periods(PATH_TO_OUTPUTS)

# # --- SAVE ---
# if df_powiat_all is not None:
#     print(f"\nFinal Powiat Panel: {df_powiat_all.shape}")
#     print(df_powiat_all['programming_period'].value_counts())
#     df_powiat_all.to_csv("Master_Powiat_Panel_With_Periods.csv", index=False)

# if df_gmina_all is not None:
#     print(f"Final Gmina Panel: {df_gmina_all.shape}")
#     df_gmina_all.to_csv("Master_Gmina_Panel_With_Periods.csv", index=False)

In [4]:
# # 1. Find files
# powiat_paths = glob.glob(os.path.join(fpath, "*powiat*.csv"))
# gmina_paths  = glob.glob(os.path.join(fpath, "*gmina*.csv"))

# print(f"Found {len(powiat_paths)} Powiat files.")
# print(f"Found {len(gmina_paths)} Gmina files.")

# # Helper to load, tag, and concat
# def load_tag_concat(file_list, level_name):
#     if not file_list:
#         print(f"No files found for {level_name}. Skipping.")
#         return None
        
#     dfs = []
#     for f in file_list:
#         print(f"  Loading: {os.path.basename(f)}...")
#         df = pd.read_csv(f, low_memory= False)
        
#         # A. Add Programming Period Column
#         period = get_period_from_filename(f)
#         df['programming_period'] = period
        
#         # B. Ensure Year is numeric
#         if 'Year' in df.columns:
#             df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)
        
#         dfs.append(df)
        
#     full_df = pd.concat(dfs, ignore_index=True)

In [5]:
powiat_files = [
    fpath+"\\powiat_level_200713.csv",
    fpath+"\\powiat_level_201420.csv",
    fpath+"\\powiat_level_202127.csv",
]

gmina_files = [
    fpath+"\\gmina_level_200713.csv",
    fpath+"\\gmina_level_201420.csv",
    fpath+"\\gmina_level_202127.csv",
]

In [ ]:
def load_single_csv(f):
    df = pd.read_csv(f, low_memory=False)
    df["programming_period"] = get_period_from_filename(f)

    if "Year" in df.columns:
        df["Year"] = (
            pd.to_numeric(df["Year"], errors="coerce")
            .fillna(0)
            .astype(int)
        )
    return df

In [12]:
df_200713_gmina = load_single_csv(gmina_files[0])
df_201420_gmina = load_single_csv(gmina_files[1])
df_202027_gmina = load_single_csv(gmina_files[2])

df_200713_powiat = load_single_csv(powiat_files[0])
df_201420_powiat = load_single_csv(powiat_files[1])
df_202027_powiat = load_single_csv(powiat_files[2])

In [14]:
df_202027_gmina = df_202027_gmina[df_202027_gmina['Year']<2026]
df_202027_powiat = df_202027_powiat[df_202027_powiat['Year']<2026]

In [55]:
sorted(pd.Series(df_200713_gmina.columns.tolist() + df_201420_gmina.columns.tolist()+ df_202027_gmina.columns.tolist()).unique())
# df_200713_powiat.dtypes
# df_201420_powiat.dtypes
# df_202027_powiat.dtypes

['EU_subsidy_PLN',
 'EU_subsidy_PLN_perc',
 'ID',
 'Year',
 'action',
 'additional_goal',
 'beneficiary',
 'city_id',
 'contractor_name',
 'dziedzina',
 'eligible_expenses_PLN',
 'end_date',
 'eu_fishing_vessel_id',
 'eu_fund',
 'euro_exchange_rate',
 'funding_type',
 'gmina',
 'gmina_id',
 'goal',
 'is_competition',
 'location_count',
 'obszar',
 'powiat',
 'powiat_id',
 'priority',
 'program',
 'programming_period',
 'project_completed',
 'project_place',
 'project_title',
 'short_desc',
 'specific_objective',
 'start_date',
 'subaction',
 'subsidy_PLN',
 'support_category',
 'terriority_type',
 'territorial_project',
 'total_value_PLN',
 'voivodeship_id',
 'voviodeship']

In [56]:

# 2. Prepare each dataframe
# (We use .copy() to avoid SettingWithCopy warnings on your original dfs)
df_p1 = df_200713_powiat.copy()
df_p2 = df_201420_powiat.copy()
df_p3 = df_202027_powiat.copy()

df_p1['programming_period'] = '2007-2013'
df_p2['programming_period'] = '2014-2020'
df_p3['programming_period'] = '2021-2027'

# 3. Rename columns & Concatenate
powiat_list = [df_p1, df_p2, df_p3]
# for df in powiat_list:
#     df.rename(columns=col_map, inplace=True)
#     # Ensure Year is int
#     if 'Year' in df.columns:
#         df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)

# 4. Stack them
df_powiat_master = pd.concat(powiat_list, ignore_index=True)

# 5. Clean up (Fill NaNs with 0 for financial columns)
fin_cols = ['total_value_PLN', 'EU_subsidy_PLN', 'eligible_expenses_PLN']
for c in fin_cols:
    if c in df_powiat_master.columns:
        df_powiat_master[c] = df_powiat_master[c].fillna(0)

# Sort and Save
df_powiat_master = df_powiat_master.sort_values(['powiat_id', 'Year', 'programming_period'])
print(f"Master Powiat Panel Created: {df_powiat_master.shape}")
df_powiat_master.to_csv(fpath+"\\Master_Powiat_Panel.csv", index=False)

Master Powiat Panel Created: (9740, 10)


In [59]:

# 2. Prepare each dataframe
# (We use .copy() to avoid SettingWithCopy warnings on your original dfs)
df_p1 = df_200713_gmina.copy()
df_p2 = df_201420_gmina.copy()
df_p3 = df_202027_gmina.copy()

df_p1['programming_period'] = '2007-2013'
df_p2['programming_period'] = '2014-2020'
df_p3['programming_period'] = '2021-2027'

# 3. Rename columns & Concatenate
powiat_list = [df_p1, df_p2, df_p3]
# for df in powiat_list:
#     df.rename(columns=col_map, inplace=True)
#     # Ensure Year is int
#     if 'Year' in df.columns:
#         df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)

# 4. Stack them
df_powiat_master = pd.concat(powiat_list, ignore_index=True)

# 5. Clean up (Fill NaNs with 0 for financial columns)
fin_cols = ['total_value_PLN', 'EU_subsidy_PLN', 'eligible_expenses_PLN']
for c in fin_cols:
    if c in df_powiat_master.columns:
        df_powiat_master[c] = df_powiat_master[c].fillna(0)

# Sort and Save
df_powiat_master = df_powiat_master.sort_values(['powiat_id', 'Year', 'programming_period'])
print(f"Master Powiat Panel Created: {df_powiat_master.shape}")
df_powiat_master.to_csv(fpath+"\\Master_Gmina_Panel.csv", index=False)

Master Powiat Panel Created: (2469078, 41)


In [40]:
display(df_200713_gmina.head(3), df_201420_gmina.head(3), df_202027_gmina.head(3))

,voivodeship_id,powiat_id,gmina_id,Year,EU_subsidy_PLN,total_value_PLN,subsidy_PLN,voviodeship,powiat,gmina,programming_period
0,2,201,201022,2007,2.267063e+06,3.330476e+06,2.267063e+06,Dolnośląskie,Powiat bolesławiecki,Bolesławiec,2007-2013
1,2,201,201022,2008,5.632393e+06,1.074916e+07,5.642517e+06,Dolnośląskie,Powiat bolesławiecki,Bolesławiec - miasto,2007-2013
2,2,201,201022,2009,1.802285e+07,4.658502e+07,1.929913e+07,Dolnośląskie,Powiat bolesławiecki,Bolesławiec - miasto,2007-2013


,project_title,short_desc,ID,beneficiary,eu_fund,program,priority,action,subaction,total_value_PLN,...,project_completed,location_count,voviodeship,powiat,voivodeship_id,powiat_id,gmina_id,city_id,Year,programming_period
0,Restoration of common culture heritage as a ba...,The main issues addressed by the project's imp...,PLBU.01.01.00-06-0195/17,GMINA PUCHACZÓW,0.0,The ENI Cross-border Cooperation Programme Pol...,1. Promotion of local culture and preservation...,1.1. Promotion of local culture and history,Brak poddziałania,162154.882353,...,Tak,2,Brest oblast,NaN,NaN,NaN,NaN,NaN,2019,Unknown
1,Restoration of common culture heritage as a ba...,The main issues addressed by the project's imp...,PLBU.01.01.00-06-0195/17,GMINA PUCHACZÓW,0.0,The ENI Cross-border Cooperation Programme Pol...,1. Promotion of local culture and preservation...,1.1. Promotion of local culture and history,Brak poddziałania,324309.764706,...,Tak,2,Brest oblast,NaN,NaN,NaN,NaN,NaN,2020,Unknown
2,Restoration of common culture heritage as a ba...,The main issues addressed by the project's imp...,PLBU.01.01.00-06-0195/17,GMINA PUCHACZÓW,0.0,The ENI Cross-border Cooperation Programme Pol...,1. Promotion of local culture and preservation...,1.1. Promotion of local culture and history,Brak poddziałania,324309.764706,...,Tak,2,Brest oblast,NaN,NaN,NaN,NaN,NaN,2021,Unknown


,project_title,short_desc,ID,beneficiary,contractor_name,eu_fishing_vessel_id,eu_fund,specific_objective,program,priority,...,voviodeship,powiat,gmina,voivodeship_id,powiat_id,gmina_id,city_id,project_completed,Year,programming_period
0,Centrum Technologii Bezpieczeństwa Publicznego,Planowany do realizacji projekt pt: Centrum Te...,FEDS.01.01-IZ.00-0003/24,Politechnika Wrocławska,APVACUUM Sp z o.o. | DSYSTEM Spółka z o. o. | ...,NaN,0.0,EFRR.CP1.I Rozwijanie i wzmacnianie zdolności ...,Fundusze Europejskie dla Dolnego Śląska 2021-2027,1.-Fundusze Europejskie na rzecz przedsiębiorc...,...,DOLNOŚLĄSKIE,Wrocław,Wrocław,2.0,264.0,264011.0,264011.0,Tak,2024,2021-2027
1,Centrum Technologii Bezpieczeństwa Publicznego,Planowany do realizacji projekt pt: Centrum Te...,FEDS.01.01-IZ.00-0003/24,Politechnika Wrocławska,APVACUUM Sp z o.o. | DSYSTEM Spółka z o. o. | ...,NaN,0.0,EFRR.CP1.I Rozwijanie i wzmacnianie zdolności ...,Fundusze Europejskie dla Dolnego Śląska 2021-2027,1.-Fundusze Europejskie na rzecz przedsiębiorc...,...,DOLNOŚLĄSKIE,Wrocław,Wrocław,2.0,264.0,264011.0,264011.0,Tak,2025,2021-2027
3,Wsparcie zielonych technologii dla przemysłu,Projekt planowany do realizacji przez Politech...,FEDS.01.01-IZ.00-0004/24,Politechnika Wrocławska,Amazemet Sp. z o.o. | Integrators Dariusz Dema...,NaN,0.0,EFRR.CP1.I Rozwijanie i wzmacnianie zdolności ...,Fundusze Europejskie dla Dolnego Śląska 2021-2027,1.-Fundusze Europejskie na rzecz przedsiębiorc...,...,DOLNOŚLĄSKIE,Wrocław,Wrocław,2.0,264.0,264011.0,264011.0,Tak,2024,2021-2027


In [45]:
sum(df_gmina_master['EU_subsidy_PLN']==0)/len(df_gmina_master)

0.99172120119332